In [69]:
import torch
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from torchvision.transforms import transforms
from torchvision.datasets import ImageFolder
from PIL import Image


In [70]:
!pip install kagglehub

In [71]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("tawsifurrahman/tuberculosis-tb-chest-xray-dataset")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\Lenovo\.cache\kagglehub\datasets\tawsifurrahman\tuberculosis-tb-chest-xray-dataset\versions\3


In [72]:
data_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize([512,512]),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

In [73]:
import os

In [74]:
class SimpleImageDataset(Dataset):
    def __init__(self,data_dir,transform = None):
        self.data = []
        self.transform = transform 
        
        self.classes = sorted([
            d for d in os.listdir(data_dir)
            if os.path.isdir(os.path.join(data_dir, d))
])
        
        self.classes_to_idx = {c:i for i, c in enumerate(self.classes)}
        for cls in self.classes:
            cls_path = os.path.join(data_dir,cls)
            if not os.path.isdir(cls_path):
                continue
            
            for img in os.listdir(cls_path):
                if img.lower().endswith(('.jpg','.jpeg','.png','.webp')):
                    self.data.append((
                        os.path.join(cls_path,img),self.classes_to_idx[cls]
                    ))
        print("classes:", self.classes)
        print("Total images:", len(self.data))
                    
    def __len__(self):
        return len(self.data)
    def __getitem__(self,idx):
        image_path,label = self.data[idx]
        image = Image.open(image_path)
        if self.transform:
            image = self.transform(image)
        return image,label

In [75]:
dataset = SimpleImageDataset(
    r'C:\Users\Lenovo\.cache\kagglehub\datasets\tawsifurrahman\tuberculosis-tb-chest-xray-dataset\versions\3\TB_Chest_Radiography_Database',
    transform=data_transforms
)

print("Dataset length:", len(dataset))


classes: ['Normal', 'Tuberculosis']
Total images: 4200
Dataset length: 4200


In [76]:
len(dataset)

4200

In [77]:
print(dataset.classes)

['Normal', 'Tuberculosis']


In [78]:
from torch.utils.data import random_split
train_dataset = int(0.8*len(dataset))
test_dataset = len(dataset) - train_dataset

train_dataset,test_dataset = random_split(dataset,[train_dataset,test_dataset])

In [79]:
train_loader = DataLoader(train_dataset, batch_size = 64, shuffle=True)
test_loader = DataLoader(test_dataset , batch_size = 64)

In [80]:
class CNNModel(nn.Module):
    def __init__(self,in_channels):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels,out_channels=16,kernel_size=3,stride = 1,padding='same'),
            nn.ReLU(),
            
            nn.MaxPool2d(kernel_size=2,stride=2),
            nn.Conv2d(16,32,kernel_size=3,stride=1,padding='same'),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2,stride=2)
            
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32*128*128,512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512,216),
            nn.ReLU(),
            nn.Dropout(0.6),
            nn.Linear(216,128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128,2)
            
        )
    def forward(self,x):
        x = self.features(x)
        y = self.classifier(x)
        return y

In [81]:
device = torch.device('cuda' if torch.cuda.is_available else 'cpu' )
model = CNNModel(1).to(device)

In [82]:
import torch.nn as nn
import torch.optim as optim

In [85]:
epochs = 2
learning_rate = 0.001
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = learning_rate)


In [86]:
model.train()
for epoch in range(epochs):
    total_loss = 0
    for image,label in train_loader:
        image,label = image.to(device),label.to(device)
        
        optimizer.zero_grad()
        pred = model(image)
        loss = criterion(pred,label)
        loss.backward()
        optimizer.step()
        total_loss +=  loss.item()
    avg_loss = total_loss/len(train_loader)
    print(f'Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}')
    
        
        

Epoch [1/2], Loss: 0.0867
Epoch [2/2], Loss: 0.0429


In [87]:
model.eval()

total = 0
correct = 0

with torch.no_grad():
    for image, label in test_loader:
        image = image.to(device)
        label = label.to(device)

        outputs = model(image)
        _, predicted = torch.max(outputs, dim=1)

        total += label.size(0)
        correct += (predicted == label).sum().item()

accuracy = correct / total
print(f"Test Accuracy: {accuracy * 100:.2f}%")


Test Accuracy: 97.98%


In [88]:
model.eval()

total = 0
correct = 0

with torch.no_grad():
    for image, label in train_loader:
        image = image.to(device)
        label = label.to(device)

        outputs = model(image)
        _, predicted = torch.max(outputs, dim=1)

        total += label.size(0)
        correct += (predicted == label).sum().item()

accuracy = correct / total
print(f"Train Accuracy: {accuracy * 100:.2f}%")


Test Accuracy: 99.55%


In [90]:
model.eval()

image_path = r'C:\Users\Lenovo\OneDrive\Desktop\Deep Learning\Pytorch\others (103).jpg'

image = Image.open(image_path)
image = data_transforms(image).unsqueeze(0).to(device)

class_names = dataset.classes   # IMPORTANT

with torch.no_grad():
    outputs = model(image)
    probs = torch.softmax(outputs, dim=1)
    predicted = torch.argmax(probs, dim=1)

predicted_class = class_names[predicted.item()]
confidence = probs[0][predicted.item()].item()

print("Predicted class:", predicted_class)
print(f"Confidence: {confidence*100:.2f}%")


Predicted class: Normal
Confidence: 96.13%


In [25]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error,accuracy_score

In [26]:
import numpy as np
import pandas as pd

# Decide number of rows
n_rows = 1000   # change this as you want

# Generate data
ages = np.random.randint(0, 101, size=n_rows)
salaries = np.random.randint(0, 100001, size=n_rows)

# Create DataFrame
df = pd.DataFrame({
    "age": ages,
    "salary": salaries
})

print(df.head())


   age  salary
0   69   24493
1   65   11631
2    1    6319
3   87   61968
4   69   97883


In [27]:
df.to_csv("age_salary_dataset.csv", index=False)


In [28]:
df

,age,salary
0,69,24493
1,65,11631
2,1,6319
3,87,61968
4,69,97883
...,...,...
995,23,7238
996,59,15100
997,7,18479
998,84,62192


In [29]:
X = df['age']
y = df['salary']

In [30]:
X_train,X_test,y_train,y_test = train_test_split(X,y,random_state=42,test_size=0.2)

In [31]:
X_train

29     95
535    74
695    30
557    80
836    72
       ..
106    73
270    45
860    47
435    71
102    77
Name: age, Length: 800, dtype: int32

In [32]:
y_train

29     29868
535    57774
695    15300
557    59572
836    28208
       ...  
106    28940
270    69783
860    94862
435    85157
102    17854
Name: salary, Length: 800, dtype: int32

In [33]:
rf = RandomForestRegressor()
rf.fit(X_train.values.reshape(-1, 1), y_train)

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [34]:
y_pred = rf.predict(X_test.values.reshape(-1,1))

In [37]:
mean_squared_error(y_test, y_pred)

935067245.4717584

In [38]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

X = df[["age"]]
y = df["salary"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

print("R² Score:", model.score(X_test, y_test))


R² Score: -0.18957111670624482


In [39]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error

# -----------------------
# Prepare data
# -----------------------
X = df[["age"]]       # feature
y = df["salary"]      # target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# -----------------------
# Scaling (MANDATORY for KNN)
# -----------------------


# -----------------------
# KNN Regressor
# -----------------------
knn = KNeighborsRegressor(
    n_neighbors=5,      # try 3,5,7,9
    weights="distance", # better than uniform
    metric="euclidean"
)

knn.fit(X_train, y_train)

# -----------------------
# Prediction & Evaluation
# -----------------------
y_pred = knn.predict(X_test)

print("R² Score:", r2_score(y_test, y_pred))
print("MAE:", mean_absolute_error(y_test, y_pred))


R² Score: -0.20913180002444265
MAE: 25611.624166666665
